# Global Food Production Stability: Spatiotemporal Analysis + XGBoost–SHAP
## Complete step-by-step notebook aligned with the latest manuscript

This notebook implements the full workflow for **Spatiotemporal Evolution and Multidimensional Nonlinear Drivers of Global Food Production Stability**. The visualization system inherits the refined Digital Connectivity notebook: Times New Roman, enlarged journal-ready fonts, framed WGS84 world maps with four-sided longitude/latitude ticks, the same five-class map palette, light-blue SHAP importance bars, viridis SHAP points, coral LOWESS curves, English variable abbreviations, restrained line weights, and 600-dpi export.

**Manuscript figures:** Fig.1 framework; Fig.2 temporal evolution; Fig.3 world maps; Fig.4 Moran/LISA/transitions; Fig.5 validation; Fig.6 global SHAP; Fig.7 nonlinear effects; Fig.8 interactions; Fig.9 development-stage heterogeneity; Fig.10 Global North–South + continental spatial heterogeneity; Fig.11 robustness.

Run sections A–L in order. After the model and SHAP objects are available, individual figure cells can be rerun without retraining.

## A. Environment, parameters and publication-style plotting system

In [ ]:
from __future__ import annotations
import os,re,gc,sys,json,math,time,shutil,zipfile,warnings,importlib.util,zlib
from pathlib import Path
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch,FancyArrowPatch
from matplotlib.colors import Normalize
from scipy.stats import randint,uniform,spearmanr
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV,TimeSeriesSplit
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
from sklearn.ensemble import RandomForestRegressor,ExtraTreesRegressor,HistGradientBoostingRegressor
from sklearn.linear_model import ElasticNet
from xgboost import XGBRegressor
import shap
from statsmodels.nonparametric.smoothers_lowess import lowess
import geopandas as gpd
import requests
from libpysal.weights import KNN
from esda.moran import Moran,Moran_Local
import joblib
from IPython.display import display

# ---------------- Paths ----------------
INPUT_FILE=Path(r'D:\工作文件\世界各国数据库v202512版（1960-2024年）.xlsx')
if not INPUT_FILE.exists():
    for p in [Path('世界各国数据库v202512版（1960-2024年）.xlsx'),Path('/mnt/data/世界各国数据库v202512版（1960-2024年）.xlsx')]:
        if p.exists(): INPUT_FILE=p; break
SHEET_NAME='原始数据'
OUTPUT_DIR=Path(r'D:\工作文件\全球粮食生产稳定性_结果_FINAL')
if str(INPUT_FILE).startswith('/mnt/data/'): OUTPUT_DIR=Path('/mnt/data/全球粮食生产稳定性_结果_FINAL')
WORLD_SHP=None

# ---------------- Study design ----------------
LONG_START_YEAR,LONG_END_YEAR=1961,2024
MODEL_START_YEAR,MODEL_END_YEAR=2000,2023
TRAIN_END_YEAR=2017
USE_LAGGED_X=True
STABILITY_WINDOW,STABILITY_MIN_OBS=10,8
ROBUST_WINDOWS=[7,10,15]
MIN_FEATURE_COVERAGE=0.60
MIN_UNIQUE_VALUES=3
MAX_PAIRWISE_CORR=0.95
ROW_MIN_FEATURE_COVERAGE=0.20
FINAL_TOP_N=30
SHAP_TOP_N=20
DEPENDENCE_TOP_N=8
INTERACTION_TOP_N=15
INTERACTION_SAMPLE_N=1200
SHAP_MAX_N=10000
LOWESS_FRAC=0.25
RANDOM_STATE=42
N_RANDOM_SEARCH=40
N_TIME_CV=4
N_JOBS=-1
MIN_COUNTRY_OBS=8
K_NEIGHBORS=6
LISA_P,LISA_PERMUTATIONS=0.05,999
MAP_YEARS=[1970,1990,2010,2024]
LISA_YEARS=[1990,2000,2010,2024]
MORAN_START_YEAR=1970
MIN_SPATIAL_N=50
MIN_HETERO_N=40

# ---------------- Publication graphics: inherited from Digital Connectivity ----------------
FIG_DPI=600
SAVE_PNG,SAVE_PDF,SAVE_TIFF=True,True,True
FIGURE_QC_ENABLED=True
MAP_LON_MIN,MAP_LON_MAX=-180.0,180.0
MAP_LAT_MIN,MAP_LAT_MAX=-60.0,88.0
MAP_LON_TICKS=[-150,-90,-30,0,30,90,150]
MAP_LAT_TICKS=[-60,-30,0,30,60]
MAP_TICK_FONTSIZE=10.8
MAP_FRAME_COLOR,MAP_FRAME_WIDTH='#666666',0.85
MAP_TICK_LENGTH,MAP_TICK_WIDTH=3.8,0.85
MAP_MISSING_COLOR,MAP_BOUNDARY_COLOR,MAP_BOUNDARY_WIDTH='#F4F4F4','#AFAFAF',0.24
MAP_COLORS=['#D7E9DF','#BFDCCF','#F1E8B3','#F3B765','#EC5428']
BAR_BLUE,LINE_RED,DARK_BLUE='#A9C9E2','#E76F51','#377EB8'
GREEN,ORANGE,PURPLE,GREY='#4DAF4A','#FF9D45','#8E63A9','#8C8C8C'
LISA_COLORS={'HH':'#D73027','HL':'#FDAE61','LH':'#74ADD1','LL':'#4575B4','NS':'#E5E5E5'}

FIG_DIR=OUTPUT_DIR/'01_Figures'; SUBFIG_DIR=FIG_DIR/'Subplots'; TABLE_DIR=OUTPUT_DIR/'02_Tables'; DATA_DIR=OUTPUT_DIR/'03_Processed_data'; MODEL_DIR=OUTPUT_DIR/'04_Model'; MAP_DIR=OUTPUT_DIR/'05_Map_cache'; CACHE_DIR=OUTPUT_DIR/'06_Cache'
for d in [OUTPUT_DIR,FIG_DIR,SUBFIG_DIR,TABLE_DIR,DATA_DIR,MODEL_DIR,MAP_DIR,CACHE_DIR]: d.mkdir(parents=True,exist_ok=True)

def set_paper_style():
    plt.rcParams.update({'font.family':'serif','font.serif':['Times New Roman','Times','Nimbus Roman','DejaVu Serif'],'mathtext.fontset':'stix','font.size':12.0,'axes.labelsize':12.0,'axes.titlesize':12.4,'xtick.labelsize':10.9,'ytick.labelsize':10.9,'legend.fontsize':10.8,'figure.titlesize':12.4,'axes.linewidth':0.90,'xtick.major.width':0.85,'ytick.major.width':0.85,'xtick.direction':'out','ytick.direction':'out','figure.facecolor':'white','axes.facecolor':'white','savefig.facecolor':'white','pdf.fonttype':42,'ps.fonttype':42,'axes.unicode_minus':False})
set_paper_style()
FIGURE_QC_REPORT=[]
def audit_figure_layout(fig,filename):
    if not FIGURE_QC_ENABLED:return {}
    fig.canvas.draw(); axes=[a for a in fig.axes if a.get_visible() and not getattr(a,'_qc_ignore',False)]
    rec={'Figure':filename,'Axes_N':len(axes),'QC':'PASS'}; FIGURE_QC_REPORT.append(rec); return rec
def save_figure(fig,filename,subplots=False):
    audit_figure_layout(fig,filename)
    if SAVE_PNG:fig.savefig(FIG_DIR/f'{filename}.png',dpi=FIG_DPI,bbox_inches='tight',pad_inches=0.02)
    if SAVE_PDF:fig.savefig(FIG_DIR/f'{filename}.pdf',bbox_inches='tight',pad_inches=0.02)
    if SAVE_TIFF:fig.savefig(FIG_DIR/f'{filename}.tif',dpi=FIG_DPI,bbox_inches='tight',pad_inches=0.02)
    if subplots:
        fig.canvas.draw(); r=fig.canvas.get_renderer()
        for i,ax in enumerate(fig.axes,1):
            if getattr(ax,'_qc_ignore',False):continue
            try:
                bb=ax.get_tightbbox(r).transformed(fig.dpi_scale_trans.inverted()).expanded(1.03,1.07)
                fig.savefig(SUBFIG_DIR/f'{filename}_panel_{i:02d}.png',dpi=FIG_DPI,bbox_inches=bb,pad_inches=0.02)
            except Exception:pass
    return fig
def style_axis(ax,grid_axis=None):
    ax.spines['top'].set_visible(False);ax.spines['right'].set_visible(False);ax.spines['left'].set_linewidth(.9);ax.spines['bottom'].set_linewidth(.9)
    if grid_axis:ax.grid(axis=grid_axis,color='#E7E7E7',linewidth=.55,alpha=.85,zorder=0)
    ax.tick_params(pad=2.5)
def _format_lon(v):
    if abs(v)<1e-9:return '0°'
    return f'{abs(v):g}°{"E" if v>0 else "W"}'
def _format_lat(v):
    if abs(v)<1e-9:return '0°'
    return f'{abs(v):g}°{"N" if v>0 else "S"}'
def style_world_map_axes(ax):
    ax.set_xlim(MAP_LON_MIN,MAP_LON_MAX);ax.set_ylim(MAP_LAT_MIN,MAP_LAT_MAX);ax.set_aspect('equal',adjustable='box');ax.margins(x=0,y=0)
    ax.set_xticks(MAP_LON_TICKS);ax.set_yticks(MAP_LAT_TICKS);ax.set_xticklabels([_format_lon(x) for x in MAP_LON_TICKS],fontsize=MAP_TICK_FONTSIZE);ax.set_yticklabels([_format_lat(y) for y in MAP_LAT_TICKS],fontsize=MAP_TICK_FONTSIZE)
    ax.tick_params(axis='x',top=True,bottom=True,labeltop=True,labelbottom=True,direction='out',length=MAP_TICK_LENGTH,width=MAP_TICK_WIDTH,pad=2.5);ax.tick_params(axis='y',left=True,right=True,labelleft=True,labelright=True,direction='out',length=MAP_TICK_LENGTH,width=MAP_TICK_WIDTH,pad=2.5)
    for s in ax.spines.values():s.set_visible(True);s.set_color(MAP_FRAME_COLOR);s.set_linewidth(MAP_FRAME_WIDTH)
    ax.grid(False);ax.set_xlabel('');ax.set_ylabel('')

FEATURE_ABBR_RULES=[('替代及核能','ANE'),('人均可耕地','ARLPC'),('年平均降水','PRE'),('森林砍伐','DEFCO2'),('年度工业淡水抽取','IFW'),('石油、天然气及煤炭发电量占比','FOS'),('制造业增加值（现价美元）','MVA'),('航空货运量','AIRF'),('安全管理式卫生设施','SAN'),('可耕地（公顷）','ARL'),('低出生体重','LBW'),('货物和服务出口','EXP'),('女性人口占总人口','FEM'),('广义货币','BM'),('个人汇款，接收','REM'),('可再生内部淡水资源总量','RWR'),('旅游服务','TRV'),('中高技术制造业增加值','MHT'),('30至70岁','NCD'),('新感染艾滋病病毒','HIV'),('非水力可再生能源发电量','NHE'),('氧化亚氮','N2O'),('中低收入经济体商品出口','XLM'),('来自高收入经济体的商品进口','IHI'),('计算机、通信和其他服务','ICTS'),('15-64岁人口','WAP'),('永久性农田','PCL'),('乙型肝炎疫苗第三剂','HEPB'),('统计绩效指标','SPI'),('水资源压力','WS'),('农业淡水抽取','AFW'),('化肥消费','FER'),('灌溉农业用地','IRR'),('农业、林业和渔业增加值','AVA'),('人均国内生产总值','GDPPC'),('政治稳定','PS'),('监管质量','RQ'),('法治','RL'),('腐败控制','CC')]
CATEGORY_PREFIX={'农业与土地':'AGR','水资源与生态环境':'ENV','人口与城镇化':'POP','经济发展与产业结构':'ECO','贸易与全球化':'TRD','金融与宏观稳定':'FIN','基础设施与数字化':'INF','制度与治理':'GOV','社会与人力资本':'SOC','能源与资源利用':'ENE','其他':'OTH'}
safe_to_orig={};orig_to_safe={}
def classify_feature(field):
    s=str(field); rules=[('农业与土地',['农业','耕地','可耕地','农田','化肥','灌溉','土地']),('水资源与生态环境',['水资源','淡水','降水','森林','排放','二氧化碳','氧化亚氮','气候']),('人口与城镇化',['人口','城镇','农村','出生','卫生设施']),('贸易与全球化',['出口','进口','贸易','汇款','旅游服务','外国直接投资']),('金融与宏观稳定',['债务','通货膨胀','利率','股票','金融','储备']),('基础设施与数字化',['互联网','宽带','移动蜂窝','铁路','航空','港口','电力接入']),('制度与治理',['政治稳定','法治','监管质量','腐败控制','政府效能']),('社会与人力资本',['教育','就业','学校','人力资本','预期寿命']),('能源与资源利用',['能源','发电','石油','煤炭','天然气']),('经济发展与产业结构',['GDP','国内生产总值','增加值','制造业','工业','服务业'])]
    for cat,keys in rules:
        if any(k in s for k in keys):return cat
    return '其他'
def field_to_abbr(field):
    s=str(field)
    for key,abbr in FEATURE_ABBR_RULES:
        if key in s:return abbr
    cat=classify_feature(s);return f'{CATEGORY_PREFIX.get(cat,"VAR")}{zlib.crc32(s.encode("utf-8"))%100:02d}'
def get_plot_abbr(feature):return field_to_abbr(safe_to_orig.get(str(feature),str(feature)))
print('Python',sys.version.split()[0],'| pandas',pd.__version__,'| xgboost',__import__('xgboost').__version__,'| shap',shap.__version__)
print('INPUT_FILE =',INPUT_FILE);print('OUTPUT_DIR =',OUTPUT_DIR)

## B. Read the full database, resolve fields and construct food-production stability

In [ ]:
CORE_CANDIDATES={'year':['年份'],'code':['国家代码'],'name':['国家名称'],'name_en':['国家英文名'],'food_index':['粮食生产指数（2014-2016年=100）'],'cereal_output':['谷物产量（公吨）'],'cereal_yield':['谷物单产（每公顷公斤）'],'crop_index':['作物生产指数（2014-2016年=100）'],'gdp_pc':['人均国内生产总值（人均GDP）（2015年不变价美元）','人均国内生产总值（2015年不变价美元）','人均国内生产总值（现价美元）']}
AGGREGATE_CODES={'AFE','AFW','ARB','CEB','CSS','EAR','EAS','EAP','TEA','EMU','ECS','ECA','TEC','EUU','FCS','HPC','HIC','IBD','IBT','IDA','IDB','IDX','LDC','LTE','LCN','LAC','TLA','LMY','LIC','LMC','MEA','MNA','TMN','MIC','NAC','OED','OSS','PRE','PSS','PST','SAS','TSA','SSA','SSF','TSS','SST','UMC','WLD','INX'}
def resolve_column(columns,candidates,required=False):
    lookup={str(c).strip():c for c in columns}
    for x in candidates:
        if x in lookup:return lookup[x]
    for x in candidates:
        hit=[c for c in columns if x in str(c)]
        if len(hit)==1:return hit[0]
    if required:raise KeyError(f'Cannot resolve required field: {candidates}')
    return None
cache=CACHE_DIR/'raw_original.parquet'
if cache.exists():raw=pd.read_parquet(cache)
else:
    engine='calamine' if importlib.util.find_spec('python_calamine') else 'openpyxl';raw=pd.read_excel(INPUT_FILE,sheet_name=SHEET_NAME,engine=engine)
    try:raw.to_parquet(cache,index=False)
    except Exception:pass
core={k:resolve_column(raw.columns,v,required=(k in ['year','code','food_index'])) for k,v in CORE_CANDIDATES.items()}
df=raw.copy();rename={core['year']:'year',core['code']:'code'}
if core.get('name'):rename[core['name']]='name'
if core.get('name_en'):rename[core['name_en']]='name_en'
df=df.rename(columns=rename);df['year']=pd.to_numeric(df.year,errors='coerce');df['code']=df.code.astype(str).str.strip().str.upper();df=df[df.year.between(LONG_START_YEAR,LONG_END_YEAR)&~df.code.isin(AGGREGATE_CODES)&(df.code.str.len()==3)].sort_values(['code','year']).reset_index(drop=True)
if 'name_en' not in df:df['name_en']=df.get('name',df.code).astype(str)
if 'name' not in df:df['name']=df.name_en
def rolling_detrended_stability(data,value_col,window=10,min_obs=8,out_col='STABILITY'):
    rows=[]
    for code,g in data[['year','code','name','name_en',value_col]].groupby('code',sort=False):
        g=g.sort_values('year').copy();years=g.year.to_numpy(float);vals=pd.to_numeric(g[value_col],errors='coerce').to_numpy(float);out=np.full(len(g),np.nan)
        for i in range(len(g)):
            lo=max(0,i-window+1);yy=years[lo:i+1];vv=vals[lo:i+1];ok=np.isfinite(vv)&(vv>0)&np.isfinite(yy)
            if ok.sum()<min_obs:continue
            x=yy[ok]-np.mean(yy[ok]);y=np.log(vv[ok]);b=np.polyfit(x,y,1);resid=y-(b[0]*x+b[1]);vol=np.std(resid,ddof=1);out[i]=1/(1+vol) if np.isfinite(vol) else np.nan
        z=g[['year','code','name','name_en']].copy();z[out_col]=out;rows.append(z)
    return pd.concat(rows,ignore_index=True)
stability_panel=rolling_detrended_stability(df,core['food_index'],STABILITY_WINDOW,STABILITY_MIN_OBS,'STABILITY');stability_panel.to_parquet(DATA_DIR/'food_production_stability_panel.parquet',index=False)
print('Rows:',len(df),'| countries:',df.code.nunique(),'| stability observations:',stability_panel.STABILITY.notna().sum());display(stability_panel.STABILITY.describe())

## C. World geometry, continents and configurable Global North–South grouping

In [ ]:
NE_URL='https://naturalearth.s3.amazonaws.com/50m_cultural/ne_50m_admin_0_countries.zip';DB_TO_NE_CODE={'XKX':'KOS'}
def download_natural_earth():
    zp=MAP_DIR/'ne_50m_admin_0_countries.zip';out=MAP_DIR/'ne_50m_admin_0_countries';shp=out/'ne_50m_admin_0_countries.shp'
    if shp.exists():return shp
    r=requests.get(NE_URL,timeout=60);r.raise_for_status();zp.write_bytes(r.content);out.mkdir(exist_ok=True)
    with zipfile.ZipFile(zp) as z:z.extractall(out)
    return shp
def _find_field(cols,cands):
    up={str(c).upper():c for c in cols}
    for x in cands:
        if x.upper() in up:return up[x.upper()]
    return None
def load_world_map():
    shp=Path(WORLD_SHP) if WORLD_SHP and Path(WORLD_SHP).exists() else download_natural_earth();w=gpd.read_file(shp);wb=_find_field(w.columns,['WB_A3','WB_A3_EH']);adm=_find_field(w.columns,['ADM0_A3','SOV_A3']);iso=_find_field(w.columns,['ISO_A3','ISO_A3_EH']);cont=_find_field(w.columns,['CONTINENT'])
    def choose(r):
        for f in [wb,adm,iso]:
            if f is None:continue
            v=str(r[f]).strip().upper()
            if len(v)==3 and v!='-99':return v
        return None
    w['GEO_CODE']=w.apply(choose,axis=1);w['CONTINENT_STD']=w[cont].astype(str) if cont else 'Unknown';w=w[w.GEO_CODE.notna()&(w.GEO_CODE!='ATA')&(w.CONTINENT_STD!='Antarctica')].copy();w=w.to_crs('EPSG:4326');bad=~w.geometry.is_valid
    if bad.any():w.loc[bad,'geometry']=w.loc[bad,'geometry'].buffer(0)
    cm=w.groupby('GEO_CODE').CONTINENT_STD.first().to_dict();w=w.dissolve(by='GEO_CODE',as_index=False);w['CONTINENT_STD']=w.GEO_CODE.map(cm);return w
world=load_world_map();continent_map=world[['GEO_CODE','CONTINENT_STD']].rename(columns={'GEO_CODE':'MAP_CODE','CONTINENT_STD':'Continent'}).drop_duplicates();continent_map['Continent_group']=continent_map.Continent.replace({'North America':'Americas','South America':'Americas'})
# Conventional editable grouping; replace with the exact published classification adopted in the manuscript if needed.
GLOBAL_NORTH_CODES=set('USA CAN AUS NZL JPN KOR ISR SGP ISL NOR SWE FIN DNK GBR IRL FRA DEU NLD BEL LUX CHE AUT ITA ESP PRT GRC CYP MLT CZE SVK POL HUN SVN HRV EST LVA LTU ROU BGR'.split())
def map_code(s):return pd.Series(s,index=getattr(s,'index',None)).astype(str).str.upper().replace(DB_TO_NE_CODE)
def north_south(code):return 'Global North' if str(code).upper() in GLOBAL_NORTH_CODES else 'Global South'
print('World geometries:',len(world))

## D. Fig. 1 — analytical framework

In [ ]:
def figure1_framework():
    fig,ax=plt.subplots(figsize=(12.8,5.7));ax.set_xlim(0,1);ax.set_ylim(0,1);ax.axis('off')
    def box(x,y,w,h,text,fc):
        p=FancyBboxPatch((x,y),w,h,boxstyle='round,pad=.012,rounding_size=.012',facecolor=fc,edgecolor='#666666',linewidth=.9);ax.add_patch(p);ax.text(x+w/2,y+h/2,text,ha='center',va='center',fontsize=11.6)
    def arrow(x1,y1,x2,y2):ax.add_patch(FancyArrowPatch((x1,y1),(x2,y2),arrowstyle='-|>',mutation_scale=12,linewidth=1.0,color='#666666'))
    box(.03,.62,.18,.18,'Long-term food\nproduction series','#D9EAF7');box(.26,.62,.18,.18,'Rolling detrended\nstability','#BDD7EE');box(.49,.62,.18,.18,'Spatiotemporal\npatterns','#D9EAF7');box(.74,.62,.22,.18,'Spatial clustering\nand persistence','#BDD7EE')
    for a,b in [(.21,.26),(.44,.49),(.67,.74)]:arrow(a,.71,b,.71)
    box(.10,.22,.20,.18,'High-dimensional\nfeature screening','#F1E8B3');box(.40,.22,.20,.18,'XGBoost + SHAP\nnonlinear effects','#F3D4A2');box(.70,.22,.20,.18,'Interactions and\nheterogeneity','#F1E8B3');arrow(.60,.62,.50,.40);arrow(.30,.31,.40,.31);arrow(.60,.31,.70,.31)
    ax.text(.5,.94,'Analytical framework for global food production stability',ha='center',va='center',fontsize=13.2,fontweight='bold');fig.subplots_adjust(left=.02,right=.98,top=.96,bottom=.03);save_figure(fig,'Fig01_Analytical_framework');plt.show();return fig
figure1_framework()

## E. Figs. 2–4 — temporal evolution, world maps and spatial dependence

In [ ]:
sp=stability_panel.copy();sp['MAP_CODE']=map_code(sp.code);sp=sp.merge(continent_map,on='MAP_CODE',how='left');sp['Continent_group']=sp.Continent_group.fillna('Unknown')
def figure2_temporal():
    glob=sp.groupby('year').STABILITY.mean();cont=sp[sp.Continent_group!='Unknown'].groupby(['year','Continent_group']).STABILITY.mean().unstack();colors={'Africa':ORANGE,'Asia':DARK_BLUE,'Europe':GREEN,'Americas':PURPLE,'Oceania':GREY};fig,ax=plt.subplots(figsize=(10.8,5.6));ax.plot(glob.index,glob.values,color='#222222',lw=2.2,label='Global')
    for c in ['Africa','Asia','Europe','Americas','Oceania']:
        if c in cont:ax.plot(cont.index,cont[c],lw=1.6,color=colors[c],label=c)
    ax.set_xlabel('Year');ax.set_ylabel('Food production stability');style_axis(ax);ax.legend(frameon=False,ncol=3);fig.subplots_adjust(left=.10,right=.98,top=.96,bottom=.13);save_figure(fig,'Fig02_Temporal_evolution');plt.show();return fig
figure2_temporal()
def pooled_breaks(panel,years):
    x=panel[panel.year.isin(years)].STABILITY.dropna().to_numpy();q=np.unique(np.quantile(x,[0,.2,.4,.6,.8,1]));return q if len(q)==6 else np.linspace(np.nanmin(x),np.nanmax(x),6)
def plot_world_value(ax,year,br):
    d=sp.loc[sp.year==year,['MAP_CODE','STABILITY']].drop_duplicates('MAP_CODE');m=world.merge(d,left_on='GEO_CODE',right_on='MAP_CODE',how='left');m.plot(ax=ax,color=MAP_MISSING_COLOR,edgecolor=MAP_BOUNDARY_COLOR,linewidth=MAP_BOUNDARY_WIDTH,aspect=1.0,zorder=1);cls=pd.cut(m.STABILITY,bins=br,labels=False,include_lowest=True)
    for k in range(5):
        p=m[cls==k]
        if len(p):p.plot(ax=ax,color=MAP_COLORS[k],edgecolor=MAP_BOUNDARY_COLOR,linewidth=MAP_BOUNDARY_WIDTH,aspect=1.0,zorder=2)
    style_world_map_axes(ax)
def figure3_maps():
    br=pooled_breaks(sp,MAP_YEARS);fig,axes=plt.subplots(2,2,figsize=(13.6,7.9))
    for ax,y,l in zip(axes.ravel(),MAP_YEARS,'abcd'):plot_world_value(ax,y,br);ax.set_title(f'({l}) {y}',loc='left',fontweight='bold',fontsize=12.1,pad=15)
    handles=[mpatches.Patch(facecolor=MAP_COLORS[i],edgecolor='none',label=f'{br[i]:.3f}–{br[i+1]:.3f}') for i in range(5)];fig.legend(handles=handles,title='Food production stability',loc='lower center',ncol=5,frameon=False,bbox_to_anchor=(.5,.005));fig.subplots_adjust(left=.055,right=.945,bottom=.105,top=.97,wspace=.12,hspace=.22);save_figure(fig,'Fig03_Global_spatial_distribution',True);plt.show();return fig
figure3_maps()
def spatial_year(year,local=False):
    d=sp.loc[sp.year==year,['MAP_CODE','STABILITY']].dropna();g=world.merge(d,left_on='GEO_CODE',right_on='MAP_CODE',how='inner')
    if len(g)<MIN_SPATIAL_N:return None
    gp=g.to_crs('EPSG:8857');cent=gp.geometry.centroid;coords=np.column_stack([cent.x,cent.y]);w=KNN.from_array(coords,k=min(K_NEIGHBORS,len(g)-1));w.transform='r';mi=Moran(g.STABILITY.to_numpy(),w,permutations=LISA_PERMUTATIONS);out={'gdf':g,'I':mi.I,'p':mi.p_sim}
    if local:
        ml=Moran_Local(g.STABILITY.to_numpy(),w,permutations=LISA_PERMUTATIONS);sig=ml.p_sim<LISA_P;lab=np.array(['NS']*len(g),dtype=object);lab[sig&(ml.q==1)]='HH';lab[sig&(ml.q==2)]='LH';lab[sig&(ml.q==3)]='LL';lab[sig&(ml.q==4)]='HL';g=g.copy();g['LISA']=lab;out['gdf']=g
    return out
mrows=[]
for y in range(MORAN_START_YEAR,LONG_END_YEAR+1):
    r=spatial_year(y)
    if r:mrows.append({'Year':y,'Moran_I':r['I'],'p_sim':r['p']})
moran_df=pd.DataFrame(mrows);lisa_dict={}
for y in LISA_YEARS:
    r=spatial_year(y,True)
    if r:lisa_dict[y]=r['gdf']
def lisa_transition(y1,y2):
    a=lisa_dict[y1][['GEO_CODE','LISA']].rename(columns={'LISA':'from'});b=lisa_dict[y2][['GEO_CODE','LISA']].rename(columns={'LISA':'to'});z=a.merge(b,on='GEO_CODE');order=['HH','HL','LH','LL','NS'];return pd.crosstab(z['from'],z['to'],normalize='index').reindex(index=order,columns=order,fill_value=0)*100
transition_2010_2024=lisa_transition(2010,2024);moran_df.to_csv(TABLE_DIR/'Global_Moran_I.csv',index=False);transition_2010_2024.to_csv(TABLE_DIR/'LISA_transition_2010_2024_pct.csv')
def plot_lisa(ax,g):
    world.plot(ax=ax,color=MAP_MISSING_COLOR,edgecolor=MAP_BOUNDARY_COLOR,linewidth=MAP_BOUNDARY_WIDTH,aspect=1.0,zorder=1)
    for k,c in LISA_COLORS.items():
        p=g[g.LISA==k]
        if len(p):p.plot(ax=ax,color=c,edgecolor=MAP_BOUNDARY_COLOR,linewidth=MAP_BOUNDARY_WIDTH,aspect=1.0,zorder=2)
    style_world_map_axes(ax)
def figure4_spatial():
    fig=plt.figure(figsize=(13.7,8.7));gs=fig.add_gridspec(2,2,height_ratios=[.75,1.05],hspace=.30,wspace=.16);ax=fig.add_subplot(gs[0,0]);ax.plot(moran_df.Year,moran_df.Moran_I,color=DARK_BLUE,lw=1.9);ax.scatter(moran_df.Year,moran_df.Moran_I,color=DARK_BLUE,s=14,zorder=3);ax.axhline(0,color=GREY,ls=':',lw=.9);ax.set_xlabel('Year');ax.set_ylabel("Global Moran's I");ax.set_title('(a) Spatial autocorrelation over time',loc='left',fontweight='bold');style_axis(ax)
    axh=fig.add_subplot(gs[0,1]);im=axh.imshow(transition_2010_2024.to_numpy(),cmap='viridis',vmin=0,vmax=np.nanmax(transition_2010_2024.to_numpy()));axh.set_xticks(range(5),transition_2010_2024.columns);axh.set_yticks(range(5),transition_2010_2024.index);axh.set_xlabel('To');axh.set_ylabel('From');axh.set_title('(b) LISA transition, 2010–2024',loc='left',fontweight='bold')
    for i in range(5):
        for j in range(5):axh.text(j,i,f'{transition_2010_2024.iloc[i,j]:.1f}',ha='center',va='center',fontsize=10.2,color='white' if transition_2010_2024.iloc[i,j]>50 else 'black')
    cb=fig.colorbar(im,ax=axh,fraction=.046,pad=.04);cb.set_label('Row percentage (%)');cb.ax.tick_params(labelsize=10.2)
    for j,y in enumerate([1990,2024]):
        axm=fig.add_subplot(gs[1,j]);plot_lisa(axm,lisa_dict[y]);axm.set_title(f'({chr(99+j)}) LISA clusters, {y}',loc='left',fontweight='bold',fontsize=12.1,pad=15)
    handles=[mpatches.Patch(facecolor=c,edgecolor='none',label=k) for k,c in LISA_COLORS.items()];fig.legend(handles=handles,loc='lower center',ncol=5,frameon=False,bbox_to_anchor=(.5,.005));fig.subplots_adjust(left=.07,right=.96,bottom=.085,top=.97);save_figure(fig,'Fig04_Moran_LISA_transition',True);plt.show();return fig
figure4_spatial()

## F. High-dimensional candidate screening and lagged model panel

In [ ]:
ID_FIELDS={'year','code','name','name_en','index'};LEAK_PATTERNS=['粮食生产指数','作物生产指数','谷物产量','谷物单产','畜牧业生产指数'];AUX_PATTERNS=['数据源数量','标准误差','置信区间','百分位排名'];candidate=[]
for c in df.columns:
    if c in ID_FIELDS:continue
    s=pd.to_numeric(df[c],errors='coerce');df[c]=s
    if any(p in str(c) for p in LEAK_PATTERNS+AUX_PATTERNS):continue
    if s.notna().sum()>0:candidate.append(c)
train_mask=df.year.between(MODEL_START_YEAR,TRAIN_END_YEAR);coverage=df.loc[train_mask,candidate].notna().mean();kept=[c for c in candidate if coverage[c]>=MIN_FEATURE_COVERAGE and df.loc[train_mask,c].nunique(dropna=True)>=MIN_UNIQUE_VALUES];corr=df.loc[train_mask,kept].corr(min_periods=40).abs();ordered=sorted(kept,key=lambda c:coverage[c],reverse=True);selected=[]
for c in ordered:
    if all((not np.isfinite(corr.loc[c,s])) or corr.loc[c,s]<MAX_PAIRWISE_CORR for s in selected):selected.append(c)
safe_to_orig={f'F{i+1:04d}':c for i,c in enumerate(selected)};orig_to_safe={v:k for k,v in safe_to_orig.items()};feature_map=pd.DataFrame({'Safe_feature':list(safe_to_orig),'Original_field':list(safe_to_orig.values())});feature_map['Category']=feature_map.Original_field.map(classify_feature);feature_map['Coverage_train']=feature_map.Original_field.map(coverage);feature_map['Plot_abbreviation']=feature_map.Original_field.map(field_to_abbr);feature_map.to_csv(TABLE_DIR/'Feature_screening_map.csv',index=False,encoding='utf-8-sig');safe_features=list(safe_to_orig)
base=df[['year','code','name','name_en']+selected].rename(columns=orig_to_safe).copy();gdp_col=core.get('gdp_pc');base['GDPPC_RAW']=pd.to_numeric(df[gdp_col],errors='coerce') if gdp_col in df.columns else np.nan;base=base.sort_values(['code','year']);prev_year=base.groupby('code').year.shift(1)
if USE_LAGGED_X:
    lag=base.groupby('code')[safe_features+['GDPPC_RAW']].shift(1);lag.loc[prev_year.ne(base.year-1),:]=np.nan;base[safe_features+['GDPPC_RAW']]=lag
lagged_feature_panel=base.copy()
def build_model_frame(target_panel,target='STABILITY'):
    z=lagged_feature_panel.merge(target_panel[['year','code',target]],on=['year','code'],how='inner');z=z[z.year.between(MODEL_START_YEAR,MODEL_END_YEAR)].copy();z=z[z[safe_features].notna().mean(axis=1)>=ROW_MIN_FEATURE_COVERAGE];good=z.groupby('code').size();z=z[z.code.isin(good[good>=MIN_COUNTRY_OBS].index)].copy();return z.sort_values(['year','code']).reset_index(drop=True)
model_frame=build_model_frame(stability_panel);print('Features after screening:',len(safe_features),'| model rows:',len(model_frame),'| countries:',model_frame.code.nunique());display(feature_map.head())

## G. Two-stage XGBoost, temporal validation and Fig. 5

In [ ]:
train0=model_frame[model_frame.year<=TRAIN_END_YEAR].copy();test0=model_frame[model_frame.year>TRAIN_END_YEAR].copy();pre_model=XGBRegressor(n_estimators=500,max_depth=4,learning_rate=.035,subsample=.85,colsample_bytree=.75,reg_lambda=2,objective='reg:squarederror',tree_method='hist',random_state=RANDOM_STATE,n_jobs=N_JOBS);pre_model.fit(train0[safe_features],train0.STABILITY);shap_train=train0 if len(train0)<=SHAP_MAX_N else train0.sample(SHAP_MAX_N,random_state=RANDOM_STATE);pre_sv=np.asarray(shap.TreeExplainer(pre_model).shap_values(shap_train[safe_features]));pre_imp=pd.DataFrame({'Safe_feature':safe_features,'Mean_abs_SHAP':np.mean(np.abs(pre_sv),axis=0)}).sort_values('Mean_abs_SHAP',ascending=False).merge(feature_map,on='Safe_feature',how='left');final_features=pre_imp.head(min(FINAL_TOP_N,len(pre_imp))).Safe_feature.tolist();pre_imp['Selected_final']=pre_imp.Safe_feature.isin(final_features);pre_imp.to_csv(TABLE_DIR/'Preliminary_SHAP_screening.csv',index=False,encoding='utf-8-sig')
def year_cv_indices(frame,n_splits=4):
    yrs=np.array(sorted(frame.year.unique()));ts=TimeSeriesSplit(n_splits=n_splits);return [(np.where(frame.year.isin(yrs[a]))[0],np.where(frame.year.isin(yrs[b]))[0]) for a,b in ts.split(yrs)]
xgb=XGBRegressor(objective='reg:squarederror',tree_method='hist',random_state=RANDOM_STATE,n_jobs=N_JOBS);params={'n_estimators':randint(350,1100),'max_depth':randint(2,7),'learning_rate':uniform(.015,.08),'subsample':uniform(.70,.28),'colsample_bytree':uniform(.60,.38),'min_child_weight':randint(1,10),'reg_alpha':uniform(0,1.5),'reg_lambda':uniform(.5,4.5)};search=RandomizedSearchCV(xgb,params,n_iter=N_RANDOM_SEARCH,scoring='neg_root_mean_squared_error',cv=year_cv_indices(train0,N_TIME_CV),random_state=RANDOM_STATE,n_jobs=1,verbose=1);search.fit(train0[final_features],train0.STABILITY);best_params=search.best_params_;final_model=XGBRegressor(objective='reg:squarederror',tree_method='hist',random_state=RANDOM_STATE,n_jobs=N_JOBS,**best_params);final_model.fit(train0[final_features],train0.STABILITY)
def metric_row(y,p,name):return {'Model':name,'N':len(y),'R2':r2_score(y,p),'RMSE':mean_squared_error(y,p)**.5,'MAE':mean_absolute_error(y,p)}
pred_xgb=final_model.predict(test0[final_features]);perf=[metric_row(test0.STABILITY,pred_xgb,'XGBoost')];models={'Random Forest':RandomForestRegressor(n_estimators=500,min_samples_leaf=2,n_jobs=N_JOBS,random_state=RANDOM_STATE),'Extra Trees':ExtraTreesRegressor(n_estimators=500,min_samples_leaf=2,n_jobs=N_JOBS,random_state=RANDOM_STATE),'HistGradientBoosting':Pipeline([('imp',SimpleImputer(strategy='median')),('m',HistGradientBoostingRegressor(random_state=RANDOM_STATE))]),'Elastic Net':Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler()),('m',ElasticNet(alpha=.001,l1_ratio=.5,max_iter=20000,random_state=RANDOM_STATE))])}
for n,m in models.items():m.fit(train0[final_features],train0.STABILITY);perf.append(metric_row(test0.STABILITY,m.predict(test0[final_features]),n))
performance=pd.DataFrame(perf).sort_values('RMSE');performance.to_csv(TABLE_DIR/'Model_performance.csv',index=False);joblib.dump(final_model,MODEL_DIR/'XGBoost_final.joblib');(MODEL_DIR/'best_params.json').write_text(json.dumps(best_params,ensure_ascii=False,indent=2),encoding='utf-8')
def figure5_model():
    fig,axes=plt.subplots(1,2,figsize=(11.8,4.9));ax=axes[0];ax.scatter(test0.STABILITY,pred_xgb,s=19,alpha=.58,color=DARK_BLUE,edgecolors='none');lo=min(test0.STABILITY.min(),pred_xgb.min());hi=max(test0.STABILITY.max(),pred_xgb.max());ax.plot([lo,hi],[lo,hi],color=LINE_RED,lw=1.7,ls='--');ax.set_xlabel('Observed stability');ax.set_ylabel('Predicted stability');ax.set_title('(a) Temporal holdout prediction',loc='left',fontweight='bold');style_axis(ax);ax=axes[1];q=performance.sort_values('R2');ax.barh(q.Model,q.R2,color=BAR_BLUE,edgecolor='none');ax.set_xlabel('Holdout $R^2$');ax.set_title('(b) Model comparison',loc='left',fontweight='bold');style_axis(ax,'x');fig.subplots_adjust(left=.09,right=.98,top=.94,bottom=.15,wspace=.28);save_figure(fig,'Fig05_Model_validation',True);plt.show();return fig
figure5_model();display(performance)

## H. Global SHAP interpretation and Figs. 6–7

In [ ]:
interpret_model=XGBRegressor(objective='reg:squarederror',tree_method='hist',random_state=RANDOM_STATE,n_jobs=N_JOBS,**best_params);interpret_model.fit(model_frame[final_features],model_frame.STABILITY);shap_sample=model_frame if len(model_frame)<=SHAP_MAX_N else model_frame.sample(SHAP_MAX_N,random_state=RANDOM_STATE);shap_sample=shap_sample.reset_index(drop=True);explainer=shap.TreeExplainer(interpret_model);shap_values=np.asarray(explainer.shap_values(shap_sample[final_features]));importance=pd.DataFrame({'Safe_feature':final_features,'Mean_abs_SHAP':np.mean(np.abs(shap_values),axis=0)});importance['Importance_share_pct']=importance.Mean_abs_SHAP/importance.Mean_abs_SHAP.sum()*100;importance=importance.merge(feature_map,on='Safe_feature',how='left').sort_values('Mean_abs_SHAP',ascending=False).reset_index(drop=True);importance.to_csv(TABLE_DIR/'SHAP_global_importance.csv',index=False,encoding='utf-8-sig')
def feature_color_values(v):
    x=np.asarray(v,float);out=np.full(len(x),.5);ok=np.isfinite(x)
    if ok.sum()<2:return out
    a,b=np.nanquantile(x[ok],[.01,.99])
    if not np.isfinite(a) or not np.isfinite(b) or b<=a:return out
    out[ok]=np.clip((x[ok]-a)/(b-a),0,1);return out
def plot_shap_combo(ax,X,sv,features,top_n=20):
    imp=np.nanmean(np.abs(sv),axis=0);order=np.argsort(-imp)[:top_n];y=np.arange(len(order));rng=np.random.default_rng(RANDOM_STATE);ax.axvline(0,color='#808080',ls=':',lw=.85,zorder=1)
    for pos,j in enumerate(order):
        jitter=np.clip(rng.normal(0,.085,len(sv)),-.22,.22);ax.scatter(sv[:,j],np.full(len(sv),pos)+jitter,c=feature_color_values(X.iloc[:,j]),cmap='viridis',vmin=0,vmax=1,s=7.0,alpha=.72,edgecolors='none',rasterized=True,zorder=3)
    ax.set_yticks(y,[get_plot_abbr(features[j]) for j in order]);ax.invert_yaxis();ax.set_xlabel('SHAP value');ax.grid(axis='y',color='#EEEEEE',lw=.5,zorder=0);style_axis(ax);top=ax.twiny();top.barh(y,imp[order],height=.58,color=BAR_BLUE,alpha=.72,edgecolor='none',zorder=0);top.set_ylim(ax.get_ylim());top.set_xlabel('Mean |SHAP|');top.xaxis.set_label_position('top');top.xaxis.tick_top();ax.set_zorder(top.get_zorder()+1);ax.patch.set_visible(False);mx=max(np.nanmax(imp[order]),1e-9);top.set_xlim(0,mx*1.17)
    for pos,j in enumerate(order):top.text(imp[j]+mx*.015,pos,f'{imp[j]:.3f}',va='center',ha='left',fontsize=10.0,color='#555555')
def figure6_shap():
    fig,ax=plt.subplots(figsize=(8.3,5.7));plot_shap_combo(ax,shap_sample[final_features],shap_values,final_features,SHAP_TOP_N);cax=fig.add_axes([.915,.18,.018,.61]);cax._qc_ignore=True;sm=mpl.cm.ScalarMappable(norm=Normalize(0,1),cmap='viridis');cb=fig.colorbar(sm,cax=cax);cb.set_ticks([0,1]);cb.set_ticklabels(['Low','High']);cb.set_label('Feature value',fontsize=10.4);cb.ax.tick_params(labelsize=10.0);fig.subplots_adjust(left=.13,right=.885,top=.86,bottom=.12);save_figure(fig,'Fig06_Global_SHAP');plt.show();return fig
figure6_shap()
def lowess_curve(x,y):
    x=np.asarray(x,float);y=np.asarray(y,float);ok=np.isfinite(x)&np.isfinite(y);x=x[ok];y=y[ok]
    if len(x)<30:return None,None
    a,b=np.quantile(x,[.01,.99]);ok=(x>=a)&(x<=b);z=lowess(y[ok],x[ok],frac=LOWESS_FRAC,it=1,return_sorted=True);q=pd.DataFrame({'x':z[:,0],'y':z[:,1]}).groupby('x',as_index=False).y.mean();return q.x.to_numpy(),q.y.to_numpy()
def zero_crossings(xs,ys,max_n=2):
    if xs is None:return []
    out=[]
    for i in range(len(xs)-1):
        if ys[i]*ys[i+1]<0:out.append(xs[i]-ys[i]*(xs[i+1]-xs[i])/(ys[i+1]-ys[i]))
    mid=np.median(xs);return sorted(sorted(out,key=lambda z:abs(z-mid))[:max_n])
def max_curvature(xs,ys):
    if xs is None or len(xs)<7:return np.nan
    dy=np.gradient(ys,xs);d2=np.gradient(dy,xs);k=np.abs(d2);edge=max(2,int(.05*len(k)));return xs[edge+np.nanargmax(k[edge:-edge])]
threshold_rows=[]
def plot_dependence(ax,f,letter):
    j=final_features.index(f);x=shap_sample[f].to_numpy(float);y=shap_values[:,j];ax.scatter(x,y,c=feature_color_values(x),cmap='viridis',vmin=0,vmax=1,s=8,alpha=.68,edgecolors='none',rasterized=True);xs,ys=lowess_curve(x,y)
    if xs is not None:
        ax.plot(xs,ys,color=LINE_RED,lw=1.8,zorder=4);zs=zero_crossings(xs,ys,2);mc=max_curvature(xs,ys)
        for n,t in enumerate(zs,1):ax.axvline(t,color='#777777',ls=':',lw=.85);threshold_rows.append({'Safe_feature':f,'Type':'SHAP_zero_crossing','Number':n,'Value':t})
        if np.isfinite(mc):threshold_rows.append({'Safe_feature':f,'Type':'maximum_curvature','Number':1,'Value':mc})
    ax.axhline(0,color='#8A8A8A',ls=':',lw=.85);ax.set_xlabel(get_plot_abbr(f));ax.set_ylabel(f'SHAP value for {get_plot_abbr(f)}');ax.set_title(letter,loc='left',fontweight='bold',fontsize=11.6,pad=3);style_axis(ax)
def figure7_dependence():
    global threshold_rows;threshold_rows=[];feats=importance.head(DEPENDENCE_TOP_N).Safe_feature.tolist();fig,axes=plt.subplots(4,2,figsize=(11.6,11.2));letters=[f'({chr(97+i)})' for i in range(len(feats))]
    for ax,f,l in zip(axes.ravel(),feats,letters):plot_dependence(ax,f,l)
    cax=fig.add_axes([.92,.20,.014,.60]);cax._qc_ignore=True;sm=mpl.cm.ScalarMappable(norm=Normalize(0,1),cmap='viridis');cb=fig.colorbar(sm,cax=cax);cb.set_ticks([0,1]);cb.set_ticklabels(['Low','High']);cb.set_label('Feature value',fontsize=10.4);cb.ax.tick_params(labelsize=10.0);fig.subplots_adjust(left=.085,right=.895,bottom=.06,top=.985,wspace=.28,hspace=.36);save_figure(fig,'Fig07_Nonlinear_SHAP_dependence',True);plt.show();t=pd.DataFrame(threshold_rows).merge(feature_map,on='Safe_feature',how='left');t.to_csv(TABLE_DIR/'SHAP_turning_points.csv',index=False,encoding='utf-8-sig');return fig,t
fig7,turning_points=figure7_dependence();display(turning_points.head(20))

## I. Fig. 8 — SHAP interactions and strongest conditional effect

In [ ]:
inter_feats=importance.head(INTERACTION_TOP_N).Safe_feature.tolist()
# IMPORTANT: interaction values are computed on ALL final model features, then subset to Top interaction features.
Xi=shap_sample[final_features].copy()
if len(Xi)>INTERACTION_SAMPLE_N:Xi=Xi.sample(INTERACTION_SAMPLE_N,random_state=RANDOM_STATE)
iv=np.asarray(shap.TreeExplainer(interpret_model).shap_interaction_values(Xi));idx=[final_features.index(f) for f in inter_feats];sub=np.abs(iv[:,idx,:][:,:,idx]).mean(axis=0);np.fill_diagonal(sub,0);pairs=[]
for i in range(len(inter_feats)):
    for j in range(i+1,len(inter_feats)):pairs.append({'Feature_1':inter_feats[i],'Feature_2':inter_feats[j],'Mean_abs_interaction':sub[i,j]})
interaction_pairs=pd.DataFrame(pairs).sort_values('Mean_abs_interaction',ascending=False).reset_index(drop=True);interaction_pairs['Field_1']=interaction_pairs.Feature_1.map(safe_to_orig);interaction_pairs['Field_2']=interaction_pairs.Feature_2.map(safe_to_orig);interaction_pairs.to_csv(TABLE_DIR/'SHAP_interaction_pairs.csv',index=False,encoding='utf-8-sig')
def figure8_interactions():
    topf=inter_feats[:10];ii=[inter_feats.index(f) for f in topf];mat=sub[np.ix_(ii,ii)];f1=interaction_pairs.iloc[0].Feature_1;f2=interaction_pairs.iloc[0].Feature_2;j1=final_features.index(f1);x=shap_sample[f1].to_numpy(float);y=shap_values[:,j1];c=shap_sample[f2].to_numpy(float);fig,axes=plt.subplots(1,2,figsize=(12.2,5.5),gridspec_kw={'width_ratios':[1.0,1.12]});ax=axes[0];im=ax.imshow(mat,cmap='viridis',aspect='auto');labs=[get_plot_abbr(f) for f in topf];ax.set_xticks(range(len(labs)),labs,rotation=45,ha='right');ax.set_yticks(range(len(labs)),labs);ax.set_title('(a) Mean absolute SHAP interaction',loc='left',fontweight='bold');cb=fig.colorbar(im,ax=ax,fraction=.046,pad=.04);cb.ax.tick_params(labelsize=10.0);ax=axes[1];ax.scatter(x,y,c=feature_color_values(c),cmap='viridis',vmin=0,vmax=1,s=8,alpha=.62,edgecolors='none',rasterized=True);med=np.nanmedian(c)
    for mask,col,lab in [(c<=med,DARK_BLUE,f'Low {get_plot_abbr(f2)}'),(c>med,LINE_RED,f'High {get_plot_abbr(f2)}')]:
        xs,ys=lowess_curve(x[mask],y[mask])
        if xs is not None:ax.plot(xs,ys,color=col,lw=1.9,label=lab)
    ax.axhline(0,color=GREY,ls=':',lw=.85);ax.set_xlabel(get_plot_abbr(f1));ax.set_ylabel(f'SHAP value for {get_plot_abbr(f1)}');ax.set_title('(b) Strongest conditional interaction',loc='left',fontweight='bold');ax.legend(frameon=False);style_axis(ax);fig.subplots_adjust(left=.08,right=.98,top=.94,bottom=.14,wspace=.28);save_figure(fig,'Fig08_SHAP_interactions',True);plt.show();return fig
figure8_interactions();display(interaction_pairs.head(15))

## J. Fig. 9 — development-stage heterogeneity

In [ ]:
cg=model_frame.groupby('code',as_index=False).GDPPC_RAW.median().dropna();cg['Development_group']=pd.qcut(cg.GDPPC_RAW,3,labels=['Lower','Middle','Higher']);model_frame=model_frame.merge(cg[['code','Development_group']],on='code',how='left');shap_sample=shap_sample.merge(cg[['code','Development_group']],on='code',how='left')
def shap_group_importance(meta_col,groups):
    rows=[]
    for grp in groups:
        pos=np.where(shap_sample[meta_col].astype(str).to_numpy()==str(grp))[0]
        if len(pos)<MIN_HETERO_N:continue
        imp=np.mean(np.abs(shap_values[pos,:]),axis=0);total=imp.sum()
        for f,v in zip(final_features,imp):rows.append({'Group':str(grp),'Safe_feature':f,'Mean_abs_SHAP':v,'Share_pct':v/total*100 if total else np.nan,'N':len(pos)})
    out=pd.DataFrame(rows);return out.merge(feature_map,on='Safe_feature',how='left') if len(out) else out
group_importance=shap_group_importance('Development_group',['Lower','Middle','Higher']);group_importance.to_csv(TABLE_DIR/'Development_group_SHAP_importance.csv',index=False,encoding='utf-8-sig')
def run_group_models():
    rows=[]
    for grp in ['Lower','Middle','Higher']:
        s=model_frame[model_frame.Development_group.astype(str)==grp];tr=s[s.year<=TRAIN_END_YEAR];te=s[s.year>TRAIN_END_YEAR]
        if len(tr)<200 or len(te)<60:continue
        m=XGBRegressor(objective='reg:squarederror',tree_method='hist',random_state=RANDOM_STATE,n_jobs=N_JOBS,**best_params);m.fit(tr[final_features],tr.STABILITY);rows.append(metric_row(te.STABILITY,m.predict(te[final_features]),grp))
    return pd.DataFrame(rows)
group_performance=run_group_models();group_performance.to_csv(TABLE_DIR/'Development_group_model_performance.csv',index=False)
def figure9_development():
    top=importance.head(15).Safe_feature.tolist();piv=group_importance[group_importance.Safe_feature.isin(top)].pivot(index='Safe_feature',columns='Group',values='Share_pct').reindex(top);cols=[c for c in ['Lower','Middle','Higher'] if c in piv.columns];piv=piv[cols];fig,axes=plt.subplots(1,2,figsize=(12.2,6.2),gridspec_kw={'width_ratios':[1.18,.82]});im=axes[0].imshow(piv.to_numpy(),cmap='viridis',aspect='auto');axes[0].set_yticks(range(len(piv)),[get_plot_abbr(f) for f in piv.index]);axes[0].set_xticks(range(len(cols)),cols);axes[0].set_title('(a) SHAP importance share by development stage',loc='left',fontweight='bold');cb=fig.colorbar(im,ax=axes[0],fraction=.046,pad=.04);cb.set_label('Importance share (%)');cb.ax.tick_params(labelsize=10.2);q=group_performance.set_index('Model').reindex(cols).dropna(how='all').reset_index();axes[1].bar(q.Model,q.R2,color=BAR_BLUE,edgecolor='none',width=.62);axes[1].set_ylabel('Holdout $R^2$');axes[1].set_title('(b) Group-specific model performance',loc='left',fontweight='bold');style_axis(axes[1],'y');fig.subplots_adjust(left=.10,right=.98,top=.94,bottom=.12,wspace=.34);save_figure(fig,'Fig09_Development_stage_heterogeneity',True);plt.show();return fig
figure9_development();display(group_importance.sort_values(['Group','Mean_abs_SHAP'],ascending=[True,False]).groupby('Group').head(8))

## K. Fig. 10 — spatial heterogeneity: Global North–South and continents

In [ ]:
shap_sample['Spatial_group']=shap_sample.code.map(north_south);shap_sample['MAP_CODE']=map_code(shap_sample.code);shap_sample=shap_sample.merge(continent_map[['MAP_CODE','Continent_group']],on='MAP_CODE',how='left');spatial_importance=shap_group_importance('Spatial_group',['Global North','Global South']);spatial_importance.to_csv(TABLE_DIR/'Global_North_South_SHAP_importance.csv',index=False,encoding='utf-8-sig');continent_groups=[x for x in ['Africa','Asia','Europe','Americas','Oceania'] if (shap_sample.Continent_group==x).sum()>=MIN_HETERO_N];continent_importance=shap_group_importance('Continent_group',continent_groups);continent_importance.to_csv(TABLE_DIR/'Continental_SHAP_importance.csv',index=False,encoding='utf-8-sig')
def figure10_spatial_heterogeneity():
    top=importance.head(12).Safe_feature.tolist();ns=spatial_importance[spatial_importance.Safe_feature.isin(top)].pivot(index='Safe_feature',columns='Group',values='Share_pct').reindex(top);y=np.arange(len(top));h=.34;fig,axes=plt.subplots(1,2,figsize=(13.2,6.3),gridspec_kw={'width_ratios':[1.05,1.15]});ax=axes[0];north=ns.get('Global North',pd.Series(0,index=top)).to_numpy();south=ns.get('Global South',pd.Series(0,index=top)).to_numpy();ax.barh(y-h/2,north,height=h,color=DARK_BLUE,label='Global North');ax.barh(y+h/2,south,height=h,color=LINE_RED,label='Global South');ax.set_yticks(y,[get_plot_abbr(f) for f in top]);ax.invert_yaxis();ax.set_xlabel('SHAP importance share (%)');ax.set_title('(a) Global North–South',loc='left',fontweight='bold');ax.legend(frameon=False);style_axis(ax,'x');piv=continent_importance[continent_importance.Safe_feature.isin(top)].pivot(index='Safe_feature',columns='Group',values='Share_pct').reindex(top);piv=piv[[c for c in continent_groups if c in piv.columns]];ax=axes[1];im=ax.imshow(piv.to_numpy(),cmap='viridis',aspect='auto');ax.set_yticks(range(len(top)),[get_plot_abbr(f) for f in top]);ax.set_xticks(range(len(piv.columns)),piv.columns,rotation=25,ha='right');ax.set_title('(b) Continental heterogeneity',loc='left',fontweight='bold');cb=fig.colorbar(im,ax=ax,fraction=.046,pad=.04);cb.set_label('Importance share (%)');cb.ax.tick_params(labelsize=10.2);fig.subplots_adjust(left=.085,right=.98,top=.94,bottom=.16,wspace=.30);save_figure(fig,'Fig10_Spatial_heterogeneity',True);plt.show();return fig
figure10_spatial_heterogeneity();display(spatial_importance.sort_values(['Group','Mean_abs_SHAP'],ascending=[True,False]).groupby('Group').head(8));display(continent_importance.sort_values(['Group','Mean_abs_SHAP'],ascending=[True,False]).groupby('Group').head(5))

## L. Fig. 11 — robustness checks and consolidated export

In [ ]:
main_rank=importance.set_index('Safe_feature').Mean_abs_SHAP.reindex(final_features)
def robustness_one(label,target_panel):
    fr=build_model_frame(target_panel,'STABILITY');tr=fr[fr.year<=TRAIN_END_YEAR];te=fr[fr.year>TRAIN_END_YEAR]
    if len(tr)<200 or len(te)<50:return None
    m=XGBRegressor(objective='reg:squarederror',tree_method='hist',random_state=RANDOM_STATE,n_jobs=N_JOBS,**best_params);m.fit(tr[final_features],tr.STABILITY);p=m.predict(te[final_features]);row=metric_row(te.STABILITY,p,label);mall=XGBRegressor(objective='reg:squarederror',tree_method='hist',random_state=RANDOM_STATE,n_jobs=N_JOBS,**best_params);mall.fit(fr[final_features],fr.STABILITY);ss=fr if len(fr)<=4000 else fr.sample(4000,random_state=RANDOM_STATE);sv=np.asarray(shap.TreeExplainer(mall).shap_values(ss[final_features]));imp=pd.Series(np.mean(np.abs(sv),axis=0),index=final_features);row['SHAP_rank_spearman_vs_main']=spearmanr(main_rank.rank(),imp.rank(),nan_policy='omit').statistic;return row
rob=[{'Model':'Food index, 10-year','N':len(test0),'R2':performance.loc[performance.Model=='XGBoost','R2'].iloc[0],'RMSE':performance.loc[performance.Model=='XGBoost','RMSE'].iloc[0],'MAE':performance.loc[performance.Model=='XGBoost','MAE'].iloc[0],'SHAP_rank_spearman_vs_main':1.0}]
for w in [7,15]:
    t=rolling_detrended_stability(df,core['food_index'],w,max(5,int(w*.75)),'STABILITY');r=robustness_one(f'Food index, {w}-year',t)
    if r:rob.append(r)
for key,label in [('cereal_output','Cereal production, 10-year'),('cereal_yield','Cereal yield, 10-year')]:
    if core.get(key):
        t=rolling_detrended_stability(df,core[key],10,8,'STABILITY');r=robustness_one(label,t)
        if r:rob.append(r)
robustness=pd.DataFrame(rob);robustness.to_csv(TABLE_DIR/'Robustness_checks.csv',index=False)
def figure11_robustness():
    fig,axes=plt.subplots(1,2,figsize=(12.2,5.0));q=robustness.sort_values('R2');axes[0].barh(q.Model,q.R2,color=BAR_BLUE,edgecolor='none');axes[0].set_xlabel('Holdout $R^2$');axes[0].set_title('(a) Predictive robustness',loc='left',fontweight='bold');style_axis(axes[0],'x');q=robustness.sort_values('SHAP_rank_spearman_vs_main');axes[1].barh(q.Model,q.SHAP_rank_spearman_vs_main,color=BAR_BLUE,edgecolor='none');axes[1].set_xlabel('Spearman $\rho$');axes[1].set_title('(b) Driver-ranking consistency',loc='left',fontweight='bold');axes[1].set_xlim(0,1.02);style_axis(axes[1],'x');fig.subplots_adjust(left=.19,right=.98,top=.93,bottom=.15,wspace=.38);save_figure(fig,'Fig11_Robustness',True);plt.show();return fig
figure11_robustness();display(robustness)
feature_map.to_csv(TABLE_DIR/'Feature_plot_abbreviations_English.csv',index=False,encoding='utf-8-sig');pd.DataFrame(FIGURE_QC_REPORT).to_csv(TABLE_DIR/'Figure_QC_report.csv',index=False)
with pd.ExcelWriter(OUTPUT_DIR/'Main_results_tables.xlsx') as xw:
    performance.to_excel(xw,'Model_performance',index=False);importance.to_excel(xw,'SHAP_importance',index=False);turning_points.to_excel(xw,'Turning_points',index=False);interaction_pairs.to_excel(xw,'Interactions',index=False);group_importance.to_excel(xw,'Development_groups',index=False);spatial_importance.to_excel(xw,'North_South',index=False);continent_importance.to_excel(xw,'Continents',index=False);robustness.to_excel(xw,'Robustness',index=False);moran_df.to_excel(xw,'Moran_I',index=False);transition_2010_2024.to_excel(xw,'LISA_transition')
summary=[f'Long period: {LONG_START_YEAR}-{LONG_END_YEAR}',f'Model period: {MODEL_START_YEAR}-{MODEL_END_YEAR}',f'Train through: {TRAIN_END_YEAR}',f'Countries in main model: {model_frame.code.nunique()}',f'Country-year observations: {len(model_frame)}',f'Candidates after rules: {len(safe_features)}',f'Final features: {len(final_features)}',f'Holdout XGBoost R2: {performance.loc[performance.Model=="XGBoost","R2"].iloc[0]:.4f}'];(OUTPUT_DIR/'RUN_SUMMARY.txt').write_text('\n'.join(summary),encoding='utf-8');print('Completed. Results saved to:',OUTPUT_DIR)